In [1]:
%pip install miditok
%pip install torch
%pip install scikit-learn joblib
%pip install huggingface-hub==0.22.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.22.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.22.2 which is incompatible.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.22.2 which is incompatible.
diffusers 0.37.0 requires huggingface-hub<2.0,>=0.34.0, but you have hugg

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!mkdir -p ./ds
!mkdir -p ./ds/vgmusic_midi
!mkdir -p ./tokenizer
!mkdir -p ./datasets
!unzip ./drive/MyDrive/midis/vgmusic_midi/chunks_dedup.zip -d ./ds/vgmusic_midi

Выходные данные были обрезаны до нескольких последних строк (5000).
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_forest_templeseq_t4_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_forest_templeseq_t4_1.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_forest_templeseq_t8_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_forest_templeseq_t8_1.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_goldenwolf_t0_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_goldenwolf_t0_1.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_goron_tribeseq_t0_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_goron_tribeseq_t1_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_hawk_and_catseq_t0_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_hawk_and_catseq_t1_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_horse_battleseq_t0_0.mid  
  inflating: ./ds/vgmusic_midi/chunks_dedup/wii_horse_battleseq_t0_2.mid  
  inflating: ./ds/vgmusic_midi/chunks_de

In [ ]:
!find ./ds/vgmusic_midi/chunks_dedup -maxdepth 1 -type f | wc -l


175176


In [10]:
from miditok import REMI, TokenizerConfig
from miditok.pytorch_data import DatasetMIDI, DataCollator
from miditok.utils import split_files_for_training
from pathlib import Path
from os.path import exists
from torch.utils.data import DataLoader, Subset, Dataset, BatchSampler
from symusic import Score
from tqdm import tqdm
import os
import torch

In [11]:
class StratifiedBatchSampler(BatchSampler):

    def __init__(self, strata_labels: list[int], batch_size: int, drop_last: bool = False, generator: torch.Generator | None = None):
        if batch_size <= 0:
            raise ValueError("batch_size must be > 0")
        if len(strata_labels) == 0:
            raise ValueError("strata_labels must be non-empty")

        self.strata_labels = list(strata_labels)
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.generator = generator

        self._indices_by_strata = {}
        for idx, stratum in enumerate(self.strata_labels):
            self._indices_by_strata.setdefault(int(stratum), []).append(idx)
        self._strata = sorted(self._indices_by_strata.keys())

    def __len__(self):
        n = len(self.strata_labels)
        if self.drop_last:
            return n // self.batch_size
        return math.ceil(n / self.batch_size)

    def __iter__(self):
        pools = {}
        for stratum, indices in self._indices_by_strata.items():
            perm = torch.randperm(len(indices), generator=self.generator).tolist()
            pools[stratum] = [indices[i] for i in perm]

        for _ in range(len(self)):
            active = [s for s in self._strata if len(pools[s]) > 0]
            if len(active) == 0:
                break

            active_perm = torch.randperm(len(active), generator=self.generator).tolist()
            active = [active[i] for i in active_perm]
            batch = []

            target_per_stratum = max(1, self.batch_size // len(active))
            for stratum in active:
                take = min(target_per_stratum, len(pools[stratum]), self.batch_size - len(batch))
                for _ in range(take):
                    batch.append(pools[stratum].pop())
                if len(batch) == self.batch_size:
                    break

            while len(batch) < self.batch_size:
                active_non_empty = [s for s in active if len(pools[s]) > 0]
                if len(active_non_empty) == 0:
                    break
                for stratum in active_non_empty:
                    if len(batch) == self.batch_size:
                        break
                    batch.append(pools[stratum].pop())

            if self.drop_last and len(batch) < self.batch_size:
                continue

            if len(batch) > 0:
                yield batch


def _bucketize_by_quantiles(values: torch.Tensor, num_bins: int) -> torch.Tensor:
    if values.numel() == 0 or num_bins <= 1:
        return torch.zeros_like(values, dtype=torch.long)

    quantiles = torch.linspace(0.0, 1.0, steps=num_bins + 1, device=values.device)
    edges = torch.quantile(values, quantiles)

    unique_boundaries = []
    for boundary in edges[1:-1].tolist():
        if len(unique_boundaries) == 0 or boundary > unique_boundaries[-1]:
            unique_boundaries.append(boundary)

    if len(unique_boundaries) == 0:
        return torch.zeros_like(values, dtype=torch.long)

    boundaries = torch.tensor(unique_boundaries, dtype=values.dtype, device=values.device)
    return torch.bucketize(values.contiguous(), boundaries, right=False)



def _resolve_subset_to_base_indices(dataset: Dataset):
    if isinstance(dataset, Subset):
        parent = _resolve_subset_to_base_indices(dataset.dataset)
        return [parent[i] for i in dataset.indices]
    return list(range(len(dataset)))



def _build_rarity_strata_labels(
    dataset: Dataset,
    freq_num_bins: int,
    strata_num_bins: int,
):
    base = dataset
    while isinstance(base, Subset):
        base = base.dataset

    if not isinstance(base, DatasetWithAttributes):
        return None

    base_indices = _resolve_subset_to_base_indices(dataset)
    if len(base_indices) == 0:
        return None

    attrs_rows = []
    for base_idx in base_indices:
        key = base.files_paths[base_idx]
        attrs = base.attributes_map.get(key)
        if attrs is None:
            raise KeyError(f"Missing attributes for sample: {key}")
        attrs_rows.append(attrs)

    attrs_tensor = torch.tensor(attrs_rows, dtype=torch.float32)
    if attrs_tensor.ndim != 2 or attrs_tensor.size(0) == 0:
        return None

    n_samples, n_attrs = attrs_tensor.shape
    rarity_score = torch.zeros(n_samples, dtype=torch.float32)

    for col in range(n_attrs):
        bins = _bucketize_by_quantiles(attrs_tensor[:, col], freq_num_bins)
        counts = torch.bincount(bins, minlength=int(bins.max().item()) + 1).float()
        rarity_score += 1.0 / counts[bins].clamp_min(1.0)

    rarity_score = rarity_score / max(n_attrs, 1)
    strata = _bucketize_by_quantiles(rarity_score, strata_num_bins)
    return strata.tolist()


In [12]:
import random
import numpy as np


class DatasetWithAttributes(Dataset):
    def __init__(
        self,
        base_dataset: Dataset,
        files_paths: list[Path],
        attributes_map: dict,
        attribute_names: list[str],
    ):
        self.base_dataset = base_dataset
        self.files_paths = [str(Path(p).resolve()) for p in files_paths]
        self.attributes_map = attributes_map
        self.attribute_names = list(attribute_names)
        self.default_num_attributes = len(self.attribute_names)

        if len(self.files_paths) != len(self.base_dataset):
            raise ValueError("files_paths length must match base_dataset length")
        if len(self.attribute_names) == 0:
            raise ValueError("attribute_names must be non-empty")

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        key = self.files_paths[idx]

        attrs = self.attributes_map.get(key)
        if attrs is None:
            raise KeyError(f"Missing attributes for sample: {key}")

        item["attributes"] = torch.tensor(attrs, dtype=torch.float32)
        return item



def load_attributes_csv(attributes_csv_path: Path):
    attributes_map = {}

    with open(attributes_csv_path, newline="") as f:
        reader = csv.DictReader(f)
        if "path" not in (reader.fieldnames or []):
            raise ValueError("attributes csv must contain 'path' column")

        attr_columns = [c for c in reader.fieldnames if c != "path"]
        if len(attr_columns) == 0:
            raise ValueError("attributes csv must contain at least one attribute column")

        for row in reader:
            abs_path = str(Path(row["path"]).resolve())
            if abs_path in attributes_map:
                raise ValueError(f"Duplicate attribute row for path: {abs_path}")
            attrs = [float(row[c]) for c in attr_columns]
            attributes_map[abs_path] = attrs

    return attributes_map, attr_columns



def validate_attribute_coverage(files_paths: list[Path], attributes_map: dict):
    resolved_files = [str(Path(p).resolve()) for p in files_paths]
    file_set = set(resolved_files)
    attr_set = set(attributes_map.keys())

    missing_files = sorted(file_set - attr_set)
    extra_rows = sorted(attr_set - file_set)

    if missing_files:
        preview = ", ".join(missing_files[:5])
        raise ValueError(
            f"Missing attributes for {len(missing_files)} MIDI files. First examples: {preview}"
        )
    if extra_rows:
        preview = ", ".join(extra_rows[:5])
        raise ValueError(
            f"Attributes CSV contains {len(extra_rows)} rows with no matching MIDI file. First examples: {preview}"
        )



def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



def _resolve_dataset_with_attributes(dataset: Dataset):
    base = dataset
    while isinstance(base, Subset):
        base = base.dataset
    if not isinstance(base, DatasetWithAttributes):
        raise TypeError("Expected DatasetWithAttributes underneath the subset chain")
    return base



def get_dataset_attribute_names(dataset: Dataset):
    return list(_resolve_dataset_with_attributes(dataset).attribute_names)



def _collate_with_attributes(base_collator):
    def collate_fn(batch):
        attrs = torch.stack([sample["attributes"] for sample in batch], dim=0)
        base_batch = [{k: v for k, v in sample.items() if k != "attributes"} for sample in batch]
        output = base_collator(base_batch)
        output["attributes"] = attrs
        return output

    return collate_fn


In [13]:
def create_tokenizer(midis_dir: Path, tokenizer_dir: Path, tokenizer_params: dict) -> REMI:

    config = TokenizerConfig(**tokenizer_params)
    tokenizer = REMI(config)

    root_dir = midis_dir.resolve()
    midi_paths = list(root_dir.glob("**/*.mid"))

    tokenizer.save_pretrained(tokenizer_dir)

    return tokenizer


def open_tokenizer(tokenizer_dir: Path) -> REMI:

    tokenizer_path = Path(tokenizer_dir, "tokenizer.json")
    if not exists(tokenizer_path):
        raise FileNotFoundError(tokenizer_path)

    tokenizer = REMI.from_pretrained(tokenizer_dir, local_files_only=True)
    return tokenizer



def create_dataset(
    midis_path: Path,
    tokenizer: REMI,
    attributes_csv_path: Path,
    subset_fraction=0.3,
    val_split=0.2,
    test_split=0.1,
    max_seq_len=512,
    seed: int = 42,
):
    if not 0 < subset_fraction <= 1.0:
        raise ValueError("subset_fraction must be in (0, 1]")
    if val_split < 0 or test_split < 0 or val_split + test_split >= 1.0:
        raise ValueError("val_split and test_split must be non-negative and sum to < 1")

    files_paths = sorted(p.resolve() for p in midis_path.glob("**/*.mid"))
    if len(files_paths) == 0:
        raise ValueError(f"No MIDI files found under {midis_path}")

    full_dataset = DatasetMIDI(
        files_paths=files_paths,
        tokenizer=tokenizer,
        max_seq_len=max_seq_len
    )

    attributes_map, attribute_names = load_attributes_csv(attributes_csv_path)
    validate_attribute_coverage(files_paths, attributes_map)

    full_dataset = DatasetWithAttributes(
        base_dataset=full_dataset,
        files_paths=files_paths,
        attributes_map=attributes_map,
        attribute_names=attribute_names,
    )

    total = len(full_dataset)
    subset_size = max(1, int(total * subset_fraction))
    subset_generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(total, generator=subset_generator).tolist()[:subset_size]
    dataset = Subset(full_dataset, indices)

    total_size = len(dataset)
    val_size = int(total_size * val_split)
    test_size = int(total_size * test_split)
    train_size = total_size - val_size - test_size

    split_generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset, test_dataset = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=split_generator,
    )

    return train_dataset, val_dataset, test_dataset



def create_dataloader(
    train_dataset,
    val_dataset,
    test_dataset,
    tokenizer: REMI,
    max_seq_len=512,
    batch_size=32,
    use_stratified_sampler: bool = True,
    frequency_bins: int = 8,
    strata_bins: int = 3,
    train_shuffle: bool = True,
    seed: int | None = None,
):

    collator = DataCollator(tokenizer.pad_token_id)
    collate_fn = _collate_with_attributes(collator)
    train_generator = None if seed is None else torch.Generator().manual_seed(seed)

    train_batch_sampler = None
    if use_stratified_sampler:
        strata_labels = _build_rarity_strata_labels(
            train_dataset,
            freq_num_bins=frequency_bins,
            strata_num_bins=strata_bins,
        )
        if strata_labels is not None and len(set(strata_labels)) > 1:
            train_batch_sampler = StratifiedBatchSampler(
                strata_labels=strata_labels,
                batch_size=batch_size,
                drop_last=False,
                generator=train_generator,
            )

    if train_batch_sampler is not None:
        train_dataloader = DataLoader(
            dataset=train_dataset,
            collate_fn=collate_fn,
            batch_sampler=train_batch_sampler,
        )
    else:
        train_dataloader = DataLoader(
            dataset=train_dataset,
            collate_fn=collate_fn,
            batch_size=batch_size,
            shuffle=train_shuffle,
            generator=train_generator,
        )

    val_dataloader = DataLoader(dataset=val_dataset, collate_fn=collate_fn, batch_size=batch_size, shuffle=False)
    test_dataloader = DataLoader(dataset=test_dataset, collate_fn=collate_fn, batch_size=batch_size, shuffle=False)

    return train_dataloader, val_dataloader, test_dataloader


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import math
import csv

In [15]:
class EncoderLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, num_layers=2, dropout=0.3):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout,
                            bidirectional=False)

        self.dropout = nn.Dropout(dropout)

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar =  nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):

        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)

        last_hidden = hidden[-1]

        mu = self.fc_mu(last_hidden)
        logvar = self.fc_logvar(last_hidden)

        return (mu, logvar)

In [16]:
class DecoderLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, num_layers=2, dropout=0.3):

        super().__init__()

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(latent_dim + embed_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout)

        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

        self.z_to_hidden = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.z_to_cell = nn.Linear(latent_dim, hidden_dim * num_layers)

    def forward(self, x, z, hidden=None, cell=None):

        batch_size, seq_len = x.size()
        embedded = self.dropout(self.embedding(x))

        z_expanded = z.unsqueeze(1).expand(-1, seq_len, -1)
        lstm_input = torch.cat([embedded, z_expanded], dim=-1)

        if hidden is None or cell is None:
            device = x.device

            hidden = self.z_to_hidden(z)
            hidden = hidden.view(self.num_layers, batch_size, self.hidden_dim)

            cell = self.z_to_cell(z)
            cell = cell.view(self.num_layers, batch_size, self.hidden_dim)

        outputs, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        logits = self.fc(outputs)
        return logits, hidden, cell

In [17]:
class VAE(nn.Module):

    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, latent_dim=64, num_layers=2, dropout=0.3):

        super().__init__()

        self.encoder = EncoderLSTM(vocab_size, embed_dim, hidden_dim, latent_dim, num_layers, dropout)
        self.decoder = DecoderLSTM(vocab_size, embed_dim, hidden_dim, latent_dim, num_layers, 0.5)
        self.latent_dim = latent_dim
        self.vocab_size = vocab_size

    def reparametrize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def forward(self, src, tgt):

        mu, logvar = self.encoder(src)
        z = self.reparametrize(mu, logvar)

        logits, _, _ = self.decoder(tgt, z)

        return logits, mu, logvar, z

    def generate(self, start_token, z=None, max_len=512, temperature=1.0, eos_id=None, sample=True):

        batch_size = start_token.size(0)
        device = start_token.device

        if z is None:
            z = torch.randn(batch_size, self.latent_dim, device=device)
        if temperature <= 0:
            raise ValueError("temperature must be > 0")

        hidden = None
        cell = None
        x = start_token.unsqueeze(1)
        generated = [start_token]
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for _ in range(max_len):
            logits, hidden, cell = self.decoder(x, z, hidden, cell)
            logits = logits[:, -1, :] / temperature
            if sample:
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).squeeze(1)
            else:
                next_token = torch.argmax(logits, dim=-1)

            if eos_id is not None:
                next_token = torch.where(finished, torch.full_like(next_token, eos_id), next_token)
                finished = finished | (next_token == eos_id)

            generated.append(next_token)
            x = next_token.unsqueeze(1)

            if eos_id is not None and torch.all(finished):
                break

        return torch.stack(generated, dim=1)


def vae_loss(logits, targets, mu, logvar, pad_id, beta=1.0):

    reconstruction_loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=pad_id,
        reduction='sum') / targets.size(0)

    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / targets.size(0)

    return reconstruction_loss + beta * kl_loss, reconstruction_loss, kl_loss


def attribute_regularization_loss(latent_vectors, attribute_values, reg_dim, delta=10.0):

    if attribute_values.dim() != 1:
        raise ValueError("attribute_values must be 1D: (batch,)")
    if reg_dim < 0 or reg_dim >= latent_vectors.size(1):
        raise ValueError(f"reg_dim={reg_dim} out of range for latent dim={latent_vectors.size(1)}")

    da = attribute_values.unsqueeze(1) - attribute_values.unsqueeze(0)
    dr = latent_vectors[:, reg_dim].unsqueeze(1) - latent_vectors[:, reg_dim].unsqueeze(0)

    return torch.mean(torch.abs(torch.tanh(delta * dr) - torch.sign(da)))



def ar_vae_loss(
    logits,
    targets,
    mu,
    logvar,
    z,
    pad_id,
    attributes,
    reg_dims,
    beta=1.0,
    gamma=1.0,
    delta=10.0,
):

    vae_total, reconstruction_loss, kl_loss = vae_loss(logits, targets, mu, logvar, pad_id, beta=beta)

    if attributes.dim() != 2:
        raise ValueError("attributes must be 2D: (batch, num_attributes)")
    if len(reg_dims) != attributes.size(1):
        raise ValueError("len(reg_dims) must match number of attribute columns")

    reg_loss = torch.tensor(0.0, device=z.device)

    for attr_idx, reg_dim in enumerate(reg_dims):
        reg_loss = reg_loss + attribute_regularization_loss(
            latent_vectors=mu,
            attribute_values=attributes[:, attr_idx],
            reg_dim=reg_dim,
            delta=delta,
        )

    reg_loss = reg_loss / max(1, len(reg_dims))
    total_loss = vae_total + gamma * reg_loss

    return total_loss, reconstruction_loss, kl_loss, reg_loss


In [18]:
def _require_batch_attributes(batch, reg_dims, device):
    if "attributes" not in batch:
        raise KeyError(
            "Batch is missing required key 'attributes'. Attribute-controlled training must use real labels."
        )

    attributes = batch["attributes"].to(device).float()
    if attributes.dim() != 2:
        raise ValueError(f"attributes must be 2D, got shape={tuple(attributes.shape)}")
    if reg_dims is None or len(reg_dims) == 0:
        raise ValueError("reg_dims must be provided when use_ar_vae=True")
    if attributes.size(1) != len(reg_dims):
        raise ValueError(
            f"Expected {len(reg_dims)} attribute columns to match reg_dims, got {attributes.size(1)}"
        )
    return attributes



def train_epoch_ar(
    model,
    dataloader,
    optimizer,
    bos_id,
    eos_id,
    pad_id,
    device,
    use_ar_vae=False,
    reg_dims=None,
    beta=1.0,
    gamma=1.0,
    delta=10.0,
):

    model.train()

    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    total_reg = 0.0

    for batch in dataloader:

        input_ids = batch["input_ids"].to(device)
        batch_size, _ = input_ids.shape

        bos = torch.full((batch_size, 1), bos_id, dtype=torch.long, device=device)
        eos = torch.full((batch_size, 1), eos_id, dtype=torch.long, device=device)

        src = torch.cat([bos, input_ids], dim=1)
        decoder_input = torch.cat([bos, input_ids], dim=1)
        target = torch.cat([input_ids, eos], dim=1)

        logits, mu, logvar, z = model(src, decoder_input)

        if use_ar_vae:
            attributes = _require_batch_attributes(batch, reg_dims, device)

            loss, recon, kl, reg = ar_vae_loss(
                logits=logits,
                targets=target,
                mu=mu,
                logvar=logvar,
                z=z,
                pad_id=pad_id,
                attributes=attributes,
                reg_dims=reg_dims,
                beta=beta,
                gamma=gamma,
                delta=delta,
            )

            total_reg += reg.item()

        else:
            loss, recon, kl = vae_loss(logits, target, mu, logvar, pad_id, beta)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    n = len(dataloader)
    return total_loss / n, total_recon / n, total_kl / n, total_reg / n


In [19]:
def validate_epoch(
    model,
    dataloader,
    bos_id,
    eos_id,
    pad_id,
    device,
    beta=1.0,
    use_ar_vae=False,
    reg_dims=None,
    gamma=1.0,
    delta=10.0,
):
    model.eval()
    total_loss = total_recon = total_kl = 0.0
    total_reg = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            batch_size, seq_len = input_ids.shape

            bos = torch.full((batch_size, 1), bos_id, dtype=torch.long, device=device)
            eos = torch.full((batch_size, 1), eos_id, dtype=torch.long, device=device)

            src = torch.cat([bos, input_ids], dim=1)
            decoder_input = torch.cat([bos, input_ids], dim=1)
            target = torch.cat([input_ids, eos], dim=1)

            logits, mu, logvar, z = model(src, decoder_input)

            if use_ar_vae:
                attributes = _require_batch_attributes(batch, reg_dims, device)

                loss, recon, kl, reg = ar_vae_loss(
                    logits=logits,
                    targets=target,
                    mu=mu,
                    logvar=logvar,
                    z=z,
                    pad_id=pad_id,
                    attributes=attributes,
                    reg_dims=reg_dims,
                    beta=beta,
                    gamma=gamma,
                    delta=delta,
                )
                total_reg += reg.item()
            else:
                loss, recon, kl = vae_loss(logits, target, mu, logvar, pad_id, beta)

            total_loss += loss.item()
            total_recon += recon.item()
            total_kl += kl.item()

    n = len(dataloader)
    return total_loss / n, total_recon / n, total_kl / n, total_reg / n


In [20]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib
from pathlib import Path
from typing import Union, List, Optional



def _collect_attribute_rows(dataset: Dataset, attribute_names: list[str]):
    base = _resolve_dataset_with_attributes(dataset)
    base_indices = _resolve_subset_to_base_indices(dataset)
    name_to_idx = {name: idx for idx, name in enumerate(base.attribute_names)}
    attr_indices = [name_to_idx[name] for name in attribute_names]

    rows = []
    for base_idx in base_indices:
        key = base.files_paths[base_idx]
        attrs = base.attributes_map[key]
        rows.append([attrs[attr_idx] for attr_idx in attr_indices])

    return np.asarray(rows, dtype=np.float32)



def fit_attribute_scaler(train_dataset: Dataset, attribute_names: list[str]):
    train_rows = _collect_attribute_rows(train_dataset, attribute_names)
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train_rows)
    return scaler



def apply_attribute_scaler_to_splits(
    train_dataset: Dataset,
    val_dataset: Dataset,
    test_dataset: Dataset,
    scaler: MinMaxScaler,
    attribute_names: list[str],
):
    base = _resolve_dataset_with_attributes(train_dataset)
    if _resolve_dataset_with_attributes(val_dataset) is not base or _resolve_dataset_with_attributes(test_dataset) is not base:
        raise ValueError("All splits must share the same DatasetWithAttributes base")

    name_to_idx = {name: idx for idx, name in enumerate(base.attribute_names)}
    attr_indices = [name_to_idx[name] for name in attribute_names]

    for key, attrs in list(base.attributes_map.items()):
        attrs_array = np.asarray(attrs, dtype=np.float32)
        scaled = scaler.transform(attrs_array[attr_indices].reshape(1, -1))[0]
        for target_idx, scaled_value in zip(attr_indices, scaled):
            attrs_array[target_idx] = float(scaled_value)
        base.attributes_map[key] = attrs_array.tolist()



def describe_dataset_split(dataset: Dataset, split_name: str):
    base = _resolve_dataset_with_attributes(dataset)
    print(f"{split_name}: {len(dataset)} samples | attributes={base.attribute_names}")


In [ ]:
# Manual CSV editing is intentionally removed.
# Keep the raw attribute file intact and validate columns by name via the helpers above.


In [21]:
def directory_min_max_pitch(midis_dir: Path):

    pitches = []
    midis = list(midis_dir.glob("**/*.mid"))

    for midi in midis:

        score = Score(midi)

        for track in score.tracks:
            for note in track.notes:
                pitches.append(note.pitch)

    return min(pitches), max(pitches)

In [ ]:
!rm ./drive/MyDrive/tokenizer/tokenizer.json

In [22]:
DATA_SEED = 42
TRAIN_SEED = 43
EVAL_SEED = 44
ATTRIBUTES_CSV_PATH = Path("./drive/MyDrive/vgmusic_midi.csv")
REQUIRED_ATTRIBUTE_NAMES = ["pitch_range", "note_density", "contour"]
SCALER_PATH = Path("./drive/MyDrive/datasets/vgmusic_scaler.joblib")

MIDIS_DIR = Path("./ds/vgmusic_midi/chunks_dedup")
TOK_DIR = Path("./drive/MyDrive/tokenizer")
CHECKPOINT_DIR = Path("./drive/MyDrive/checkpoint/ar_vae_vgmusic")
OPTIMIZER_DIR = CHECKPOINT_DIR / "optimizer"
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_ar_vae.pt"
BEST_OPTIMIZER_PATH = OPTIMIZER_DIR / "best_ar_vae_optimizer.pt"
TRAINING_HISTORY_PATH = CHECKPOINT_DIR / "training_history.csv"

set_global_seed(DATA_SEED)
MIN_PITCH, MAX_PITCH = directory_min_max_pitch(Path(MIDIS_DIR))

TOKENIZER_PARAMS = {
    "pitch_range": (MIN_PITCH, MAX_PITCH),
    "beat_res": {(0, 4): 8, (4, 12): 4},
    "special_tokens": ["PAD", "BOS", "EOS"],
    "use_velocities": False,
    "use_chords": False,
    "use_rests": True,
    "use_tempos": True,
    "use_time_signatures": False,
    "use_programs": False,
    "use_pitch_intervals": True,
    "num_tempos": 32,
    "tempo_range": (40, 250),
    "remove_duplicated_notes": True,
}

if not exists(Path(TOK_DIR, "tokenizer.json").resolve()):
    print("Creating new tokenizer...")
    tokenizer = create_tokenizer(MIDIS_DIR, TOK_DIR, TOKENIZER_PARAMS)
else:
    print("Opening tokenizer...")
    tokenizer = open_tokenizer(TOK_DIR)


config.json not found in /content/drive/MyDrive/tokenizer


Opening tokenizer...


In [ ]:
set_global_seed(DATA_SEED)
train_dataset, val_dataset, test_dataset = create_dataset(
    MIDIS_DIR,
    tokenizer,
    attributes_csv_path=ATTRIBUTES_CSV_PATH,
    subset_fraction=1.0,
    val_split=0.2,
    test_split=0.1,
    seed=DATA_SEED,
)

attribute_names = get_dataset_attribute_names(train_dataset)
if attribute_names != REQUIRED_ATTRIBUTE_NAMES:
    raise ValueError(
        f"Attribute schema mismatch. Expected {REQUIRED_ATTRIBUTE_NAMES}, got {attribute_names}"
    )
'''
attribute_scaler = fit_attribute_scaler(train_dataset, REQUIRED_ATTRIBUTE_NAMES)
apply_attribute_scaler_to_splits(
    train_dataset,
    val_dataset,
    test_dataset,
    attribute_scaler,
    REQUIRED_ATTRIBUTE_NAMES,
)
joblib.dump({"attribute_names": REQUIRED_ATTRIBUTE_NAMES, "scaler": attribute_scaler}, SCALER_PATH)
'''
describe_dataset_split(train_dataset, "train")
describe_dataset_split(val_dataset, "val")
describe_dataset_split(test_dataset, "test")

torch.save(train_dataset, "./drive/MyDrive/datasets/train_dataset.pt")
torch.save(val_dataset, "./drive/MyDrive/datasets/val_dataset.pt")
torch.save(test_dataset, "./drive/MyDrive/datasets/test_dataset.pt")


In [28]:
train_dataset = torch.load("./drive/MyDrive/datasets/train_dataset.pt", weights_only=False)
val_dataset = torch.load("./drive/MyDrive/datasets/val_dataset.pt", weights_only=False)
test_dataset = torch.load("./drive/MyDrive/datasets/test_dataset.pt", weights_only=False)
attribute_names = get_dataset_attribute_names(train_dataset)
if attribute_names != REQUIRED_ATTRIBUTE_NAMES:
    raise ValueError(f"Loaded datasets contain unexpected attributes: {attribute_names}")

reg_dims = [0, 1, 2]

In [ ]:
from scipy.stats import spearmanr


def compute_loader_control_gap(model, dataloader, reg_dims, bos_id, device):
    latent_batches = []
    attribute_batches = []
    was_training = model.training
    model.eval()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            batch_size = input_ids.size(0)
            bos = torch.full((batch_size, 1), bos_id, dtype=torch.long, device=device)
            src = torch.cat([bos, input_ids], dim=1)
            mu, _ = model.encoder(src)

            latent_batches.append(mu.cpu().numpy())
            attribute_batches.append(batch["attributes"].cpu().numpy())

    if was_training:
        model.train()

    if not latent_batches:
        return float("nan")

    z = np.concatenate(latent_batches, axis=0)
    attrs = np.concatenate(attribute_batches, axis=0)
    signed_corr = np.zeros((z.shape[1], attrs.shape[1]), dtype=np.float64)

    for dim_idx in range(z.shape[1]):
        for attr_idx in range(attrs.shape[1]):
            rho, _ = spearmanr(z[:, dim_idx], attrs[:, attr_idx])
            signed_corr[dim_idx, attr_idx] = 0.0 if rho is None or np.isnan(rho) else float(rho)

    abs_corr = np.abs(signed_corr)
    gaps = []
    for attr_idx, reg_dim in enumerate(reg_dims):
        target_corr = float(abs_corr[reg_dim, attr_idx])
        other_dims = [dim for dim in range(abs_corr.shape[0]) if dim != reg_dim]
        max_leakage = float(np.max(abs_corr[other_dims, attr_idx])) if other_dims else 0.0
        gaps.append(target_corr - max_leakage)

    return float(np.mean(gaps)) if gaps else float("nan")


set_global_seed(TRAIN_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.vocab_size
pad_id = tokenizer.pad_token_id
bos_id = tokenizer._tokens_to_ids(["BOS_None"])[0]
eos_id = tokenizer._tokens_to_ids(["EOS_None"])[0]

beta_start = 0.0
beta_end = 0.15
beta_warmup_epochs = 10
gamma = 15.0
delta = 10.0

train_loader, val_loader, test_loader = create_dataloader(
    train_dataset,
    val_dataset,
    test_dataset,
    tokenizer,
    use_stratified_sampler=True,
    batch_size=32,
    frequency_bins=16,
    strata_bins=6,
    seed=TRAIN_SEED,
)

model = VAE(vocab_size, embed_dim=256, hidden_dim=256, latent_dim=64, num_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OPTIMIZER_DIR.mkdir(parents=True, exist_ok=True)

model.train()
num_epochs = 30
best_control_gap = -float("inf")
best_epoch = None
history_rows = []

for epoch in range(num_epochs):
    beta = min(beta_end, beta_start + (beta_end - beta_start) * (epoch + 1) / max(beta_warmup_epochs, 1))

    train_loss, train_recon, train_kl, train_reg = train_epoch_ar(
        model,
        train_loader,
        optimizer,
        bos_id,
        eos_id,
        pad_id,
        device,
        reg_dims=reg_dims,
        use_ar_vae=True,
        beta=beta,
        gamma=gamma,
        delta=delta,
    )

    val_loss, val_recon, val_kl, val_reg = validate_epoch(
        model,
        val_loader,
        bos_id,
        eos_id,
        pad_id,
        device,
        beta=beta,
        reg_dims=reg_dims,
        use_ar_vae=True,
        gamma=gamma,
        delta=delta,
    )

    val_control_gap = compute_loader_control_gap(model, val_loader, reg_dims, bos_id, device)
    history_rows.append({
        "epoch": epoch + 1,
        "beta": beta,
        "train_loss": train_loss,
        "train_recon": train_recon,
        "train_kl": train_kl,
        "train_reg": train_reg,
        "val_loss": val_loss,
        "val_recon": val_recon,
        "val_kl": val_kl,
        "val_reg": val_reg,
        "val_control_gap": val_control_gap,
    })

    print(
        f"Epoch {epoch+1:2d} | "
        f"Beta: {beta:.3f} | "
        f"Train Loss: {train_loss:.3f} (Recon: {train_recon:.3f} KL: {train_kl:.3f} Reg: {train_reg:.3f}) | "
        f"Val Loss: {val_loss:.3f} (Recon: {val_recon:.3f} KL: {val_kl:.3f} Reg: {val_reg:.3f}) | "
        f"Val ControlGap: {val_control_gap:.3f}"
    )

    epoch_model_path = CHECKPOINT_DIR / f"ar_vae_epoch{epoch+1}.pt"
    epoch_optimizer_path = OPTIMIZER_DIR / f"ar_vae_epoch{epoch+1}.pt"
    torch.save(model.state_dict(), epoch_model_path)
    torch.save(optimizer.state_dict(), epoch_optimizer_path)

    if np.isfinite(val_control_gap) and val_control_gap > best_control_gap:
        best_control_gap = val_control_gap
        best_epoch = epoch + 1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        torch.save(optimizer.state_dict(), BEST_OPTIMIZER_PATH)
        print(f"  Saved new best checkpoint: {BEST_MODEL_PATH} (epoch={best_epoch}, control_gap={best_control_gap:.3f})")

training_history = pd.DataFrame(history_rows)
training_history.to_csv(TRAINING_HISTORY_PATH, index=False)
print(f"Best checkpoint: {BEST_MODEL_PATH} | epoch={best_epoch} | val_control_gap={best_control_gap:.3f}")


In [ ]:
model_path = BEST_MODEL_PATH
if not model_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {model_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = REMI.from_pretrained(TOK_DIR, local_files_only=True)
vocab_size = tokenizer.vocab_size
pad_id = tokenizer.pad_token_id
bos_id = tokenizer._tokens_to_ids(["BOS_None"])[0]
eos_id = tokenizer._tokens_to_ids(["EOS_None"])[0]

model = VAE(vocab_size, embed_dim=256, hidden_dim=256, latent_dim=64, num_layers=2)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)

print(f"Loaded checkpoint: {model_path}")
model.eval()

In [29]:
train_dataset = torch.load("./drive/MyDrive/datasets/train_dataset.pt", weights_only=False)
val_dataset = torch.load("./drive/MyDrive/datasets/val_dataset.pt", weights_only=False)
test_dataset = torch.load("./drive/MyDrive/datasets/test_dataset.pt", weights_only=False)

_, _, test_loader = create_dataloader(
    train_dataset,
    val_dataset,
    test_dataset,
    tokenizer,
    use_stratified_sampler=False,
    batch_size=32,
    seed=EVAL_SEED,
)



In [25]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import metrics
from attrubute import compute_attributes_for_track


In [ ]:
from scipy.stats import spearmanr


def compute_control_metrics(z, attrs, reg_dims, attr_names):
    if len(reg_dims) != attrs.shape[1]:
        raise ValueError("len(reg_dims) must match the number of attribute columns")

    signed_corr = np.zeros((z.shape[1], attrs.shape[1]), dtype=np.float64)
    for dim_idx in range(z.shape[1]):
        for attr_idx in range(attrs.shape[1]):
            rho, _ = spearmanr(z[:, dim_idx], attrs[:, attr_idx])
            signed_corr[dim_idx, attr_idx] = 0.0 if rho is None or np.isnan(rho) else float(rho)

    abs_corr = np.abs(signed_corr)
    rows = []
    for attr_idx, (reg_dim, attr_name) in enumerate(zip(reg_dims, attr_names)):
        target_corr = float(abs_corr[reg_dim, attr_idx])
        other_dims = [dim for dim in range(abs_corr.shape[0]) if dim != reg_dim]
        max_leakage = float(np.max(abs_corr[other_dims, attr_idx])) if other_dims else 0.0
        rows.append({
            "attribute": attr_name,
            "target_dim": reg_dim,
            "target_abs_spearman": target_corr,
            "max_leakage": max_leakage,
            "control_gap": target_corr - max_leakage,
        })

    return pd.DataFrame(rows), signed_corr


z_mu, attrs = metrics.collect_latent_metric_data(model, test_loader, device, use_mu=True)
metric_summary = metrics.all_disentanglement_metrics(z_mu, attrs, attr_names=attribute_names)
control_df, signed_corr = compute_control_metrics(z_mu, attrs, reg_dims, attribute_names)

print("Disentanglement metrics on mu:")
for key, value in metric_summary.items():
    print(f"{key}: {value:.6f}")

print("\nControl metrics by designated latent dim:")
print(control_df)


Disentanglement metrics on mu:
interpretability: 0.864400
mig: 0.205847
modularity: 0.733394
sap: 0.749702
spearman: 0.947579

Control metrics by designated latent dim:
      attribute  target_dim  target_abs_spearman  max_leakage  control_gap
0   pitch_range           0             0.931963     0.359950     0.572013
1  note_density           1             0.992672     0.349250     0.643422
2       contour           2             0.918100     0.344024     0.574077


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from symusic import Score
from pathlib import Path
from scipy.stats import spearmanr


def compute_attributes_from_tokens(tokens, tokenizer, pitch_diff):
    if len(tokens) == 0:
        return {name: np.nan for name in attribute_names}

    score = tokenizer.decode([tokens])
    if not score.tracks or len(score.tracks[0].notes) == 0:
        return {name: np.nan for name in attribute_names}

    track = score.tracks[0]
    attrs = compute_attributes_for_track(score, track, pitch_diff)
    return attrs


def collect_mu_bases(model, dataloader, bos_id, device, max_bases: int | None = None):
    mu_batches = []
    collected = 0
    was_training = model.training
    model.eval()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            batch_size = input_ids.size(0)
            bos = torch.full((batch_size, 1), bos_id, dtype=torch.long, device=device)
            src = torch.cat([bos, input_ids], dim=1)
            mu, _ = model.encoder(src)

            mu_cpu = mu.cpu()
            mu_batches.append(mu_cpu)
            collected += mu_cpu.size(0)
            if max_bases is not None and collected >= max_bases:
                break

    if was_training:
        model.train()

    if not mu_batches:
        raise ValueError("Could not collect latent bases from the evaluation dataloader")

    base_latents = torch.cat(mu_batches, dim=0)
    if max_bases is not None:
        base_latents = base_latents[:max_bases]
    return base_latents


def generate_from_z(model, z, bos_id, eos_id, temperature=1.0, sample=False):
    with torch.no_grad():
        start_token = torch.full((1,), bos_id, dtype=torch.long, device=device)
        tokens = model.generate(
            start_token,
            z=z,
            max_len=max_len,
            temperature=temperature,
            eos_id=eos_id,
            sample=sample,
        )

    tokens = tokens[0].cpu().tolist()
    if tokens and tokens[0] == bos_id:
        tokens = tokens[1:]
    if eos_id in tokens:
        tokens = tokens[:tokens.index(eos_id)]
    return tokens


def traverse_dim(model, z_base, dim_idx, bos_id, eos_id, min_val=-5.0, max_val=5.0, steps=7, sample=False):
    values = np.linspace(min_val, max_val, steps)
    all_tokens = []
    for val in values:
        z = z_base.clone()
        z[0, dim_idx] = val
        tokens = generate_from_z(model, z, bos_id, eos_id, temperature=temp, sample=sample)
        all_tokens.append(tokens)
    return all_tokens, values


def monotonicity_score(sequence):
    if len(sequence) < 2:
        return np.nan
    diffs = np.diff(sequence)
    return float(max(np.mean(diffs > 0), np.mean(diffs <= 0)))


def evaluate_traversal_control(model, bos_id, eos_id, reg_dims, attr_names, base_latents, sample=False):
    summaries = []
    for dim_idx, attr_name in zip(reg_dims, attr_names):
        per_base_abs_rho = []
        per_base_mono = []

        for z_base in base_latents:
            z_base = z_base.unsqueeze(0).to(device)
            tokens_list, values = traverse_dim(
                model,
                z_base,
                dim_idx,
                bos_id,
                eos_id,
                min_val=-3.0,
                max_val=3.0,
                steps=num_steps,
                sample=sample,
            )

            attr_vals = []
            for tokens in tokens_list:
                attrs = compute_attributes_from_tokens(
                    tokens,
                    tokenizer,
                    tokenizer.config.pitch_range[1] - tokenizer.config.pitch_range[0],
                )
                attr_vals.append(attrs[attr_name])

            if np.isnan(attr_vals).any():
                continue

            rho, _ = spearmanr(values, attr_vals)
            per_base_abs_rho.append(0.0 if rho is None or np.isnan(rho) else abs(float(rho)))
            per_base_mono.append(monotonicity_score(attr_vals))

        summaries.append({
            "attribute": attr_name,
            "target_dim": dim_idx,
            "mean_abs_spearman": float(np.mean(per_base_abs_rho)) if per_base_abs_rho else np.nan,
            "median_abs_spearman": float(np.median(per_base_abs_rho)) if per_base_abs_rho else np.nan,
            "mean_monotonicity": float(np.mean(per_base_mono)) if per_base_mono else np.nan,
            "median_monotonicity": float(np.median(per_base_mono)) if per_base_mono else np.nan,
            "evaluated_bases": len(per_base_abs_rho),
        })

    return pd.DataFrame(summaries)


set_global_seed(EVAL_SEED)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.vocab_size
pad_id = tokenizer.pad_token_id
bos_id = tokenizer._tokens_to_ids(["BOS_None"])[0]
eos_id = tokenizer._tokens_to_ids(["EOS_None"])[0]

print(f"bos id: {bos_id}")
print(f"eos id: {eos_id}")
print(tokenizer._ids_to_tokens([eos_id]))

num_steps = 6
temp = 1.0
max_len = 256
num_eval_bases = 64
output_dir = Path("./traversal_demo")
output_dir.mkdir(exist_ok=True)

base_latents = collect_mu_bases(model, test_loader, bos_id, device, max_bases=num_eval_bases)
summary_df = evaluate_traversal_control(
    model,
    bos_id,
    eos_id,
    reg_dims,
    attribute_names,
    base_latents=base_latents,
    sample=False,
)
print(summary_df)

z_demo = base_latents[0].unsqueeze(0).to(device)
print("Using z_demo encoded from the first test example.")
for dim_idx, attr_name in zip(reg_dims, attribute_names):
    print(f"\n=== Traversal по измерению {dim_idx} (атрибут: {attr_name}) ===")
    tokens_list, values = traverse_dim(
        model,
        z_demo,
        dim_idx,
        bos_id,
        eos_id,
        min_val=-3.0,
        max_val=3.0,
        steps=num_steps,
        sample=False,
    )

    attr_vals = []
    for val, tokens in zip(values, tokens_list):
        midi_path = output_dir / f"dim{dim_idx}_val{val:.2f}.mid"
        score = tokenizer.decode([tokens])
        score.dump_midi(str(midi_path))

        attrs = compute_attributes_from_tokens(
            tokens,
            tokenizer,
            tokenizer.config.pitch_range[1] - tokenizer.config.pitch_range[0],
        )
        attr_val = attrs[attr_name]
        attr_vals.append(attr_val)
        print(f"  val = {val:.2f} -> {attr_name} = {attr_val:.3f}")

    plt.figure()
    plt.plot(values, attr_vals, 'o-')
    plt.xlabel(f'Значение измерения {dim_idx}')
    plt.ylabel(attr_name)
    plt.title(f'Traversal по измерению {dim_idx}')
    plt.grid(True)
    plt.savefig(output_dir / f"dim{dim_idx}_plot.png")
    plt.show()

print("\nГотово! MIDI-файлы и графики сохранены в", output_dir)
